# Wafer Map Defect Classification: Part 3, Improved Training (Augmentation + Class Weights)
**Before you run:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

When the upload button appears, select **both** zips at once (hold Ctrl): **results.zip** from Part 1 and **results2.zip** from Part 2. Only results.zip is required; results2.zip just adds the Part 2 numbers to the final table.

What changes from Part 2 (same data splits, same architectures):
1. **On-the-fly rotation and flip augmentation**: every batch sees each wafer at a random 90° rotation and flip. Defect types don't depend on orientation, so this is free extra data.
2. **Class-weighted loss**: rare classes (Donut, Scratch, Near-full) count more, so they stop being pushed into Loc.
3. **Longer training**: patience 10 (was 6), up to 60 epochs.
4. Both the **Baseline CNN** and the **Hybrid** are retrained with this identical recipe, so the comparison stays fair.
5. Grad-CAM shows the **most confident** correct wafer for each class instead of the first one found.

Run time is about 25–35 minutes on a T4.

In [ ]:
import os, time, zipfile, shutil
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf, keras
from keras import layers as L
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from google.colab import files
up = files.upload()                        # select results.zip (and results2.zip if you have it)
for f in up: zipfile.ZipFile(f).extractall('results2_old' if 'results2' in f else '.')

SEED = 42; np.random.seed(SEED); tf.random.set_seed(SEED)
CLASSES = ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Near-full', 'Random', 'Scratch', 'none']
IMG, NC = 64, 9
os.makedirs('results3', exist_ok=True)

d = np.load('results/splits_64.npz')
Xtr_i, y_tr = d['X_tr'], d['y_tr']          # integer maps (0,1,2); one-hot is done inside the pipeline
onehot = lambda a: np.eye(3, dtype=np.float32)[a]
X_va, y_va = onehot(d['X_va']), d['y_va']
X_te, y_te = onehot(d['X_te']), d['y_te']
print('train', Xtr_i.shape, 'val', X_va.shape, 'test', X_te.shape, '| GPU:', tf.config.list_physical_devices('GPU'))

## 1. Augmentation pipeline and class weights

In [ ]:
BS = 128
def aug(x, y):
    x = tf.image.rot90(x[..., None], k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    x = tf.cond(tf.random.uniform([]) < 0.5, lambda: tf.image.flip_left_right(x), lambda: x)
    return tf.one_hot(tf.cast(x[..., 0], tf.int32), 3), y

train_ds = (tf.data.Dataset.from_tensor_slices((Xtr_i, y_tr))
            .shuffle(len(y_tr), seed=SEED, reshuffle_each_iteration=True)
            .map(aug, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BS).prefetch(tf.data.AUTOTUNE))

# 'balanced' weights, square-rooted so rare classes get a boost without wrecking precision
cw = compute_class_weight('balanced', classes=np.arange(NC), y=y_tr)
class_weight = {i: float(np.sqrt(w)) for i, w in enumerate(cw)}
pd.DataFrame({'train_count': np.bincount(y_tr, minlength=NC), 'weight': [round(class_weight[i], 3) for i in range(NC)]}, index=CLASSES)

## 2. Models (identical to Part 1 and Part 2)

In [ ]:
@keras.saving.register_keras_serializable()
class PositionEmbedding(L.Layer):
    def build(self, input_shape):
        self.pos = self.add_weight(name='pos', shape=(1, input_shape[1], input_shape[2]), initializer='random_normal')
    def call(self, x):
        return x + self.pos

def transformer_block(x, dim, heads=4, mlp=256, drop=0.1, name=None):
    h = L.LayerNormalization(epsilon=1e-6)(x)
    h = L.MultiHeadAttention(num_heads=heads, key_dim=dim // heads, dropout=drop, name=name)(h, h)
    x = L.Add()([x, h])
    h = L.LayerNormalization(epsilon=1e-6)(x)
    h = L.Dense(mlp, activation='gelu')(h); h = L.Dropout(drop)(h)
    h = L.Dense(dim)(h); h = L.Dropout(drop)(h)
    return L.Add()([x, h])

def conv_bn(x, f, k=3, s=1, act=True, name=None):
    x = L.Conv2D(f, k, strides=s, padding='same', use_bias=False, name=name)(x)
    x = L.BatchNormalization()(x)
    return L.ReLU()(x) if act else x

def build_cnn():                                   # same as Part 1 baseline
    inp = L.Input((IMG, IMG, 3)); x = inp
    for f in [32, 64, 128, 256]:
        x = L.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = L.BatchNormalization()(x)
        x = L.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = L.MaxPooling2D()(x); x = L.Dropout(0.25)(x)
    x = L.GlobalAveragePooling2D()(x)
    x = L.Dense(128, activation='relu')(x); x = L.Dropout(0.4)(x)
    return keras.Model(inp, L.Dense(NC, activation='softmax')(x), name='Baseline_CNN_v2')

def build_hybrid(dim=128, depth=2):                # same as Part 2 hybrid
    inp = L.Input((IMG, IMG, 3))
    x = conv_bn(inp, 32); x = conv_bn(x, 32); x = L.MaxPooling2D()(x)
    x = conv_bn(x, 64); x = conv_bn(x, 64); x = L.MaxPooling2D()(x)
    x = conv_bn(x, dim); x = conv_bn(x, dim, name='last_conv'); x = L.MaxPooling2D()(x)
    x = L.Reshape((-1, dim))(x)
    x = PositionEmbedding()(x)
    for i in range(depth): x = transformer_block(x, dim, name=f'mhsa_{i}')
    x = L.LayerNormalization(epsilon=1e-6)(x)
    x = L.GlobalAveragePooling1D()(x); x = L.Dropout(0.3)(x)
    return keras.Model(inp, L.Dense(NC, activation='softmax')(x), name='Hybrid_CNN_Transformer_v2')

## 3. Train both with the improved recipe
Each epoch line prints only when the epoch finishes, so a pause after "Epoch 1/60" is normal.

In [ ]:
def run(model, epochs=60, lr=1e-3):
    model.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay=1e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    cb = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)]
    t0 = time.time()
    h = model.fit(train_ds, validation_data=(X_va, y_va), epochs=epochs, class_weight=class_weight, callbacks=cb, verbose=2)
    tt = (time.time() - t0) / 60
    yp = model.predict(X_te, batch_size=256, verbose=0).argmax(1)
    fn = f'results3/{model.name}.keras'; model.save(fn)
    _ = model.predict(X_te[:64], batch_size=1, verbose=0)
    t0 = time.time(); _ = model.predict(X_te[:500], batch_size=1, verbose=0)
    ms = (time.time() - t0) / 500 * 1000
    row = {'model': model.name, 'params': int(model.count_params()), 'size_MB': round(os.path.getsize(fn) / 1e6, 2),
           'test_accuracy': round(accuracy_score(y_te, yp), 4), 'macro_F1': round(f1_score(y_te, yp, average='macro'), 4),
           'inference_ms_per_wafer': round(ms, 3), 'train_time_min': round(tt, 1)}
    pd.DataFrame(classification_report(y_te, yp, labels=list(range(NC)), target_names=CLASSES, digits=4, output_dict=True, zero_division=0)).T \
        .to_csv(f'results3/table_{model.name}_per_class.csv')
    print(row); return model, h, yp, row

results, preds, hists = [], {}, {}
for builder in [build_cnn, build_hybrid]:
    keras.backend.clear_session(); tf.random.set_seed(SEED)
    m, h, yp, row = run(builder())
    results.append(row); preds[m.name] = yp; hists[m.name] = h.history
hybrid = m

## 4. Comparison: Part 2 recipe vs improved recipe

In [ ]:
old = 'results2_old/table_final_comparison.csv'
prev = pd.read_csv(old) if os.path.exists(old) else pd.read_csv('results/table_model_comparison.csv')
table = pd.concat([prev, pd.DataFrame(results)], ignore_index=True)
table.to_csv('results3/table_final_comparison_v2.csv', index=False)
table

In [ ]:
f1df = pd.DataFrame({n: f1_score(y_te, yp, average=None, labels=range(NC)) for n, yp in preds.items()}, index=CLASSES)
oldf = 'results2_old/table_per_class_f1_all_models.csv'
if os.path.exists(oldf):
    o = pd.read_csv(oldf, index_col=0)
    f1df.insert(0, 'Hybrid (Part 2)', o['Hybrid_CNN_Transformer'].values)
    f1df.insert(0, 'Baseline CNN (Part 1)', o['Baseline CNN'].values)
f1df.round(4).to_csv('results3/table_per_class_f1_v2.csv')
ax = f1df.plot.bar(figsize=(12, 4.2), width=0.8)
ax.set_ylim(0.5, 1.0); ax.set_ylabel('F1-score'); ax.set_title('Per-class F1-score before and after improved training')
plt.xticks(rotation=0); plt.legend(loc='lower right', ncol=4, fontsize=8); plt.tight_layout()
plt.savefig('results3/fig_per_class_f1_v2.png', dpi=300); plt.show()
f1df.round(3)

In [ ]:
cm = confusion_matrix(y_te, preds[hybrid.name], normalize='true')
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Proposed Hybrid CNN-Transformer: normalised confusion matrix')
plt.tight_layout(); plt.savefig('results3/fig_hybrid_confusion_matrix_v2.png', dpi=300); plt.show()

plt.figure(figsize=(7, 4))
for n, hh in hists.items(): plt.plot(hh['val_accuracy'], label=n)
plt.xlabel('Epoch'); plt.ylabel('Validation accuracy'); plt.legend(); plt.title('Validation accuracy (improved recipe)')
plt.tight_layout(); plt.savefig('results3/fig_val_accuracy_v2.png', dpi=300); plt.show()

## 5. Grad-CAM on the improved hybrid
For each class, the correctly classified test wafer the model is most confident about.

In [ ]:
grad_model = keras.Model(hybrid.inputs, [hybrid.get_layer('last_conv').output, hybrid.output])

def gradcam(img, cls=None):
    x = tf.convert_to_tensor(img[None].astype('float32'))
    with tf.GradientTape() as tape:
        conv, pred = grad_model(x)
        if cls is None: cls = int(tf.argmax(pred[0]))
        score = pred[:, cls]
    g = tape.gradient(score, conv)
    w = tf.reduce_mean(g, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * w, axis=-1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return tf.image.resize(cam[..., None], (IMG, IMG)).numpy()[..., 0], cls, float(pred[0, cls])

yp = preds[hybrid.name]
prob = hybrid.predict(X_te, batch_size=256, verbose=0)
fig, ax = plt.subplots(2, 9, figsize=(20, 5))
for c, name in enumerate(CLASSES):
    ok = np.where((y_te == c) & (yp == c))[0]
    i = ok[np.argmax(prob[ok, c])] if len(ok) else np.where(y_te == c)[0][0]
    cam, pc, p = gradcam(X_te[i])
    wafer = X_te[i].argmax(-1)
    ax[0, c].imshow(wafer, cmap='viridis'); ax[0, c].set_title(name); ax[0, c].axis('off')
    ax[1, c].imshow(wafer, cmap='gray'); ax[1, c].imshow(cam, cmap='jet', alpha=0.5)
    ax[1, c].set_title(f'{CLASSES[pc]} ({p:.2f})', fontsize=9); ax[1, c].axis('off')
ax[0, 0].text(-20, 32, 'Wafer', rotation=90, va='center'); ax[1, 0].text(-20, 32, 'Grad-CAM', rotation=90, va='center')
plt.tight_layout(); plt.savefig('results3/fig_gradcam_hybrid_v2.png', dpi=300); plt.show()

wrong = np.where(yp != y_te)[0][:6]
if len(wrong):
    fig, ax = plt.subplots(1, len(wrong), figsize=(3 * len(wrong), 3.2)); ax = np.atleast_1d(ax)
    for a, i in zip(ax, wrong):
        cam, pc, p = gradcam(X_te[i])
        a.imshow(X_te[i].argmax(-1), cmap='gray'); a.imshow(cam, cmap='jet', alpha=0.5)
        a.set_title(f'True {CLASSES[y_te[i]]}\nPred {CLASSES[pc]} ({p:.2f})', fontsize=9); a.axis('off')
    plt.tight_layout(); plt.savefig('results3/fig_gradcam_errors_v2.png', dpi=300); plt.show()

## 6. Download everything

In [ ]:
shutil.make_archive('results3', 'zip', 'results3')
files.download('results3.zip')